In [104]:
import argparse
import pandas as pd
from scipy.interpolate import interp1d

# ----------------------------------------------------------------------
def filter(input_path: str, output_path: str | None) -> None:

#    df = pd.read_csv(input_path, delim_whitespace=True)
    df = pd.read_csv(input_path, sep=r'\s+')
    #df = df.apply(lambda col: col.str.strip() if col.dtype == "object" else col)
    df = df.apply(lambda col: col.str.strip() if pd.api.types.is_string_dtype(col) else col)
    # Now read in the quenching factors (QF) for protons and carbon nuclei.
    # For deuterons assume half of protons.
    # For alphas assume 1/4th of protons.
    # For all nuclei assume equal to carbon. (which is a lower limit)
    response_c = LightOutputPorC('QF_c.csv')
    response_p = LightOutputPorC('QF_p.csv')

    #y_result = interp_c.get_light(x_query)    
    # Compute weighted energy contributions
    df['light'] = 0.0

    # Protons: full energy
    mask_p = df['ParticleName'] == 'proton'
    df.loc[mask_p, 'light'] = response_p.get_light(df.loc[mask_p, 'E(MeV)'])

    # Deuterons at detector 1: half energy
    mask_d = (df['detector#'] == 1) & (df['ParticleName'] == 'deuteron')
    df.loc[mask_d, 'light'] = 0.5 * response_p.get_light(df.loc[mask_d, 'E(MeV)'])

    # Alphas at detector 1: quarter energy
    mask_a = (df['detector#'] == 1) & (df['ParticleName'].str.contains('alpha|He'))
    df.loc[mask_a, 'light'] = 0.25 * response_p.get_light(df.loc[mask_a, 'E(MeV)'])

    # anything heavier
#    mask_heavy = (df['detector#'] == 1) & ('C' in df['ParticleName'] | 'B' in df['ParticleName'] | 'L' in df['ParticleName'])
    mask_heavy = (df['detector#'] == 1) & (df['ParticleName'].str.contains('C|B|L'))
    df.loc[mask_heavy, 'light'] = response_c.get_light(df.loc[mask_heavy, 'E(MeV)'])

    # Sum weighted energies by EventID and detector#
    sums = (
        df
        .groupby(['EventID', 'detector#'])['light']
        .sum()
        .reset_index()
    )

    # Pivot to have one column per detector
    pivot = (
        sums
        .pivot(index='EventID', columns='detector#', values='light')
        .fillna(0)
        .reset_index()
    )

    # Rename columns for clarity
    pivot.columns = ['EventID', 'E_det0(photons)', 'E_det1(photons)']

# **Filter** to only those events where detector 1 has E > 0.6 MeV
    #filtered = pivot[pivot['E_det1(MeV)'] > 0.6]

    filtered = pivot[
    (pivot['E_det1(photons)'] < 50) &
    (pivot['E_det0(photons)'].between(600, 100000))
    ]

#    filtered = pivot[pivot['EventID']==34976468]

    # Write to TSV

    # --- Output ----------------------------------------------------------
    if output_path:
        filtered.to_csv(output_path, sep='\t', index=False)
        print(f"Saved results to {output_path}")
    else:
        print(filtered.to_string(index=False))

# ----------------------------------------------------------------------

In [76]:
class LightOutputPorC:
    def __init__(self, filename):
        # Read and prepare the DataFrame
        self.df = pd.read_csv(filename) # use data from "Light output response of KamLAND liquid scintillator for
                                        #protons and 12C nuclei"
        self.df.columns = self.df.columns.str.strip()
        self.df = self.df.sort_values(by='x')
        
        # Create the interpolation function
        self.interp_func = interp1d(
            self.df['x'], 
            self.df['y'], 
            kind='linear', 
            fill_value='extrapolate'
        )

    def get_light(self, energy):
        light_output_per_mev = 1e+4 # light output for EJ-301, per MeV, for electrons
        return self.interp_func(energy)*light_output_per_mev*energy  # photons
'''
interp_c = LightOutputPorC('QF_c.csv')
interp_p = LightOutputPorC('QF_p.csv')

x_query = 7
y_result = interp_c.get_light(x_query)
print(f"y({x_query}) = {y_result}")
'''

'\ninterp_c = LightOutputPorC(\'QF_c.csv\')\ninterp_p = LightOutputPorC(\'QF_p.csv\')\n\nx_query = 7\ny_result = interp_c.get_light(x_query)\nprint(f"y({x_query}) = {y_result}")\n'

In [105]:
filter('test.dat','')

Empty DataFrame
Columns: [EventID, E_det0(photons), E_det1(photons)]
Index: []
